In [ ]:
%load_ext autoreload
%autoreload 2

# 00 · Run the M5 pipeline end-to-end

Thin demo notebook that drives `src/m5` rather than implementing logic inline.
Heavy lifting lives in the package; this notebook is just a runnable script.

Prereqs: `make bootstrap && make prep`.

In [1]:
from pathlib import Path
import pandas as pd
from m5.config import SETTINGS, set_global_seed
from m5.cv import stats_cv, lgbm_cv
from m5.evaluation import compute_components, wrmsse_for_models

set_global_seed()
long_path = SETTINGS.processed_dir / 'long.parquet'
df = pd.read_parquet(long_path)
df.head()

/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsforecast/models.py:3806: SyntaxWarning: invalid escape sequence '\h'
  """CrostonClassic model.
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsforecast/models.py:3984: SyntaxWarning: invalid escape sequence '\h'
  """CrostonOptimized model.
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsforecast/models.py:4132: SyntaxWarning: invalid escape sequence '\h'
  """CrostonSBA model.
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsforecast/models.py:4468: SyntaxWarning: invalid escape sequence '\e'
  """TSB model.


,unique_id,item_id,dept_id,cat_id,store_id,state_id,d,y,ds,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1541,0.0,2015-04-18,11512,none,none,none,none,0,0,0,2.24
1,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1542,1.0,2015-04-19,11512,none,none,none,none,0,0,0,2.24
2,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1543,0.0,2015-04-20,11512,none,none,none,none,0,0,0,2.24
3,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1544,0.0,2015-04-21,11512,none,none,none,none,0,0,0,2.24
4,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1545,0.0,2015-04-22,11512,none,none,none,none,0,0,0,2.24


## Statistical baselines (Theta + AutoETS + SeasonalNaive)

In [2]:
cv_stats = stats_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
components = compute_components(df[df['ds'] < cv_stats['ds'].min()])
stats_scores = wrmsse_for_models(cv_stats[['unique_id','ds','y']], cv_stats, components)
stats_scores

09:24:16 | INFO    | m5.cv:stats_cv:31 - stats_cv: h=28 n_windows=3 step=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/Git

AutoETS          0.863081
Theta            0.869511
SeasonalNaive    1.111490
Name: wrmsse, dtype: float64

## LightGBM global model

In [3]:
cv_lgbm = lgbm_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
lgbm_scores = wrmsse_for_models(cv_lgbm[['unique_id','ds','y']], cv_lgbm, components)
lgbm_scores

09:28:47 | INFO    | m5.cv:lgbm_cv:53 - lgbm_cv: h=28 n_windows=3 step=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:683: UserWarning: The following series were dropped completely due to the transformations and features: ['FOODS_3_595_CA_1', 'FOODS_3_595_CA_3', 'HOUSEHOLD_1_020_WI_2', 'HOUSEHOLD_1_278_CA_3', 'HOUSEHOLD_1_311_CA_2', 'HOUSEHOLD_1_386_WI_1', ...].
These series won't show up if you use `MLForecast.forecast_fitted_values()`.
You can set `dropna=False` or use transformations that require less samples to mitigate this
  warnings.warn(
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag14, lag28, rolling_mean_lag1_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag14, lag28, rolling_mean_lag1_window_size28.
  warni

LGBM    0.840123
Name: wrmsse, dtype: float64

## Combined leaderboard

In [4]:
leaderboard = pd.concat([stats_scores, lgbm_scores]).sort_values()
leaderboard.to_frame('WRMSSE')

,WRMSSE
LGBM,0.840123
AutoETS,0.863081
Theta,0.869511
SeasonalNaive,1.111490
